In [1]:
import pandas as pd

In [2]:
games = pd.read_csv('../data/games_2021_26.csv')

Formula for (POSS) possessions a game

In [3]:
games['POSS'] = games['FGA'] + 0.44 * games['FTA'] - games['OREB'] + games['TOV']

In [4]:
games[['FGA', 'FTA', 'OREB', 'TOV', 'POSS']].head()

,FGA,FTA,OREB,TOV,POSS
0,76,30,2,14,101.20
1,91,22,9,13,104.68
2,103,15,18,10,101.60
3,88,23,10,10,98.12
4,98,9,12,8,97.96


Formula for ORTG (Offensive Rating)

In [5]:
games['ORTG'] = 100 * games['PTS'] / games['POSS']

In [6]:
games[['PTS', 'POSS', 'ORTG']].head()

,PTS,POSS,ORTG
0,116,101.20,114.624506
1,120,104.68,114.635078
2,109,101.60,107.283465
3,118,98.12,120.260905
4,88,97.96,89.832585


Formula for (OPP_PTS) Opponents points

In [7]:
games['OPP_PTS'] = games['PTS'] - games['PLUS_MINUS']

Formula for (DRTG) defensive rating

In [8]:
games['DRTG'] = 100 * games['OPP_PTS'] / games['POSS']

In [9]:
games[['PTS', 'PLUS_MINUS', 'OPP_PTS', 'DRTG']].head()

,PTS,PLUS_MINUS,OPP_PTS,DRTG
0,116,7.0,109.0,107.707510
1,120,-4.0,124.0,118.456248
2,109,-7.0,116.0,114.173228
3,118,12.0,106.0,108.030982
4,88,-50.0,138.0,140.873826


Sort each team's games by date (needed for rolling averages)

In [10]:
games['GAME_DATE'] = pd.to_datetime(games['GAME_DATE'])

In [11]:
games.columns

Index(['SEASON_ID', 'TEAM_ID', 'TEAM_ABBREVIATION', 'TEAM_NAME', 'GAME_ID',
       'GAME_DATE', 'MATCHUP', 'WL', 'MIN', 'PTS', 'FGM', 'FGA', 'FG_PCT',
       'FG3M', 'FG3A', 'FG3_PCT', 'FTM', 'FTA', 'FT_PCT', 'OREB', 'DREB',
       'REB', 'AST', 'STL', 'BLK', 'TOV', 'PF', 'PLUS_MINUS', 'POSS', 'ORTG',
       'OPP_PTS', 'DRTG'],
      dtype='str')

In [12]:
games = games.sort_values(['TEAM_ABBREVIATION', 'GAME_DATE'])

Previous game stats (no leakage)

Each row is a game we're pretending to predict, so its features can only use games played before it.
`shift(1)` moves each team's stats down one row, so every game sees the team's previous game instead of its own.
`groupby` on the team keeps one team's games from spilling into the next team's rows.
A team's first game has no previous game, so it shows NaN.

In [13]:
games['PREV_ORTG'] = games.groupby(['TEAM_ABBREVIATION', 'SEASON_ID'])['ORTG'].shift(1)

In [14]:
games[['TEAM_ABBREVIATION', 'GAME_DATE', 'ORTG', 'PREV_ORTG']].head(12)

,TEAM_ABBREVIATION,GAME_DATE,ORTG,PREV_ORTG
2432,ATL,2021-10-21,107.660061,NaN
2407,ATL,2021-10-23,97.336066,107.660061
2366,ATL,2021-10-25,119.327074,97.336066
2340,ATL,2021-10-27,109.114249,119.327074
2324,ATL,2021-10-28,109.727165,109.114249
2288,ATL,2021-10-30,91.832747,109.727165
2260,ATL,2021-11-01,125.853242,91.832747
2244,ATL,2021-11-03,105.263158,125.853242
2216,ATL,2021-11-04,100.574713,105.263158
2194,ATL,2021-11-06,111.301370,100.574713


10-game rolling ORTG

Average ORTG over each team's previous 10 games. The first 10 games are NaN.

In [15]:
games['ORTG_L10'] = games.groupby(['TEAM_ABBREVIATION', 'SEASON_ID'])['PREV_ORTG'].transform(lambda x: x.rolling(10).mean())

In [16]:
games[['TEAM_ABBREVIATION', 'GAME_DATE', 'PREV_ORTG', 'ORTG_L10']].head(12)

,TEAM_ABBREVIATION,GAME_DATE,PREV_ORTG,ORTG_L10
2432,ATL,2021-10-21,NaN,NaN
2407,ATL,2021-10-23,107.660061,NaN
2366,ATL,2021-10-25,97.336066,NaN
2340,ATL,2021-10-27,119.327074,NaN
2324,ATL,2021-10-28,109.114249,NaN
2288,ATL,2021-10-30,109.727165,NaN
2260,ATL,2021-11-01,91.832747,NaN
2244,ATL,2021-11-03,125.853242,NaN
2216,ATL,2021-11-04,105.263158,NaN
2194,ATL,2021-11-06,100.574713,NaN


In [17]:
games['PREV_DRTG'] = games.groupby(['TEAM_ABBREVIATION', 'SEASON_ID'])['DRTG'].shift(1)

10-game rolling DRTG

Average DRTG over each team's previous 10 games. The first 10 games are NaN.

In [18]:
games['DRTG_L10'] = games.groupby(['TEAM_ABBREVIATION', 'SEASON_ID'])['PREV_DRTG'].transform(lambda x: x.rolling(10).mean())

In [19]:
games[['TEAM_ABBREVIATION', 'GAME_DATE', 'PREV_DRTG', 'DRTG_L10']].head(12)

,TEAM_ABBREVIATION,GAME_DATE,PREV_DRTG,DRTG_L10
2432,ATL,2021-10-21,NaN,NaN
2407,ATL,2021-10-23,82.888720,NaN
2366,ATL,2021-10-25,103.483607,NaN
2340,ATL,2021-10-27,101.721440,NaN
2324,ATL,2021-10-28,105.905006,NaN
2288,ATL,2021-10-30,120.601028,NaN
2260,ATL,2021-11-01,119.187182,NaN
2244,ATL,2021-11-03,118.387372,NaN
2216,ATL,2021-11-04,114.035088,NaN
2194,ATL,2021-11-06,119.047619,NaN


10-game rolling REB

Average rebounds over each team's previous 10 games in the same season. The first 10 games of each season are NaN.

In [20]:
games['PREV_REB'] = games.groupby(['TEAM_ABBREVIATION', 'SEASON_ID'])['REB'].shift(1)

In [21]:
games['REB_L10'] = games.groupby(['TEAM_ABBREVIATION', 'SEASON_ID'])['PREV_REB'].transform(lambda x: x.rolling(10).mean())

In [22]:
games[['TEAM_ABBREVIATION', 'GAME_DATE', 'PREV_REB', 'REB_L10']].head(12)

,TEAM_ABBREVIATION,GAME_DATE,PREV_REB,REB_L10
2432,ATL,2021-10-21,NaN,NaN
2407,ATL,2021-10-23,55.0,NaN
2366,ATL,2021-10-25,54.0,NaN
2340,ATL,2021-10-27,49.0,NaN
2324,ATL,2021-10-28,55.0,NaN
2288,ATL,2021-10-30,43.0,NaN
2260,ATL,2021-11-01,49.0,NaN
2244,ATL,2021-11-03,47.0,NaN
2216,ATL,2021-11-04,52.0,NaN
2194,ATL,2021-11-06,32.0,NaN


In [23]:
games['WIN'] = (games['WL'] == 'W').astype(int)

In [24]:
games['PREV_WIN'] = games.groupby(['TEAM_ABBREVIATION', 'SEASON_ID'])['WIN'].shift(1)

win % over the previous 10 games.

In [25]:
games['WIN_L10'] = games.groupby(['TEAM_ABBREVIATION', 'SEASON_ID'])['PREV_WIN'].transform(lambda x: x.rolling(10).mean())

In [26]:
games[['TEAM_ABBREVIATION', 'GAME_DATE', 'WIN', 'WIN_L10']].head(12)

,TEAM_ABBREVIATION,GAME_DATE,WIN,WIN_L10
2432,ATL,2021-10-21,1,NaN
2407,ATL,2021-10-23,0,NaN
2366,ATL,2021-10-25,1,NaN
2340,ATL,2021-10-27,1,NaN
2324,ATL,2021-10-28,0,NaN
2288,ATL,2021-10-30,0,NaN
2260,ATL,2021-11-01,1,NaN
2244,ATL,2021-11-03,0,NaN
2216,ATL,2021-11-04,0,NaN
2194,ATL,2021-11-06,0,NaN


In [27]:
games['PREV_DATE'] = games.groupby(['TEAM_ABBREVIATION', 'SEASON_ID'])['GAME_DATE'].shift(1)

Rest days

Days between this game and the team's previous game in the same season. 1 means a back-to-back. The first game of each season is NaN.

In [28]:
games['REST_DAYS'] = (games['GAME_DATE'] - games['PREV_DATE']).dt.days

In [29]:
games[['TEAM_ABBREVIATION', 'GAME_DATE', 'PREV_DATE', 'REST_DAYS']].head()

,TEAM_ABBREVIATION,GAME_DATE,PREV_DATE,REST_DAYS
2432,ATL,2021-10-21,NaT,NaN
2407,ATL,2021-10-23,2021-10-21,2.0
2366,ATL,2021-10-25,2021-10-23,2.0
2340,ATL,2021-10-27,2021-10-25,2.0
2324,ATL,2021-10-28,2021-10-27,1.0


In [30]:
games['HOME'] = (games['MATCHUP'].str.contains('vs.')).astype(int)

In [31]:
games['HOME'].value_counts()

HOME
0    6160
1    6140
Name: count, dtype: int64

In [32]:
home_ids = games[games['HOME'] == 1]['GAME_ID']
len(home_ids)

6140

Neutral Site games

In [33]:
games['NEUTRAL'] = (~games['GAME_ID'].isin(home_ids)).astype(int)
games['NEUTRAL'].value_counts()

NEUTRAL
0    12280
1       20
Name: count, dtype: int64

Net rating (NET_L10)

ORTG_L10 minus DRTG_L10. Positive means the team has outscored opponents per 100 possessions over its last 10 games.

In [34]:
games['NET_L10'] = (games['ORTG_L10'] - games['DRTG_L10'])